# खर्च दावा विश्लेषण

यह नोटबुक दिखाता है कि कैसे एजेंट बनाए जाएं जो प्लगइन्स का उपयोग करके स्थानीय रसीद छवियों से यात्रा खर्चों को संसाधित करते हैं, एक खर्च दावा ईमेल उत्पन्न करते हैं, और खर्च डेटा को एक पाई चार्ट का उपयोग करके विज़ुअलाइज़ करते हैं। एजेंट कार्य संदर्भ के आधार पर गतिशील रूप से फ़ंक्शन चुनते हैं।

चरण:
1. OCR एजेंट स्थानीय रसीद छवि को संसाधित करता है और यात्रा खर्च डेटा निकालता है।
2. ईमेल एजेंट एक खर्च दावा ईमेल उत्पन्न करता है।

### यात्रा खर्च परिदृश्य का उदाहरण:
कल्पना करें कि आप एक कर्मचारी हैं जो दूसरे शहर में एक व्यवसाय बैठक के लिए यात्रा कर रहे हैं। आपकी कंपनी की नीति है कि सभी उचित यात्रा संबंधित खर्चों की प्रतिपूर्ति की जाए। संभावित यात्रा खर्चों का विस्तृत विवरण निम्नलिखित है:
- परिवहन:
अपने गृह शहर से गंतव्य शहर तक राउंड ट्रिप के लिए एयरफेयर।
एयरपोर्ट तक और एयरपोर्ट से टैक्सी या राइड-हेलिंग सेवाएँ।
गंतव्य शहर में स्थानीय परिवहन (जैसे सार्वजनिक परिवहन, किराये की कारें, या टैक्सी)।

- आवास:
बैठक स्थल के करीब एक मध्यम श्रेणी के व्यावसायिक होटल में तीन रातों का ठहराव।

- भोजन:
कंपनी की प्रति दिन नीति के आधार पर नाश्ता, दोपहर का भोजन, और रात के खाने के लिए दैनिक भोजन भत्ता।

- विविध खर्चे:
एयरपोर्ट पर पार्किंग शुल्क।
होटल में इंटरनेट एक्सेस शुल्क।
टिप्स या छोटे सेवा शुल्क।

- प्रलेखन:
आप सभी रसीदें (फ्लाइट्स, टैक्सी, होटल, भोजन आदि) और प्रतिपूर्ति के लिए एक पूर्ण खर्च रिपोर्ट जमा करते हैं।


## आवश्यक पुस्तकालय आयात करें

नोटबुक के लिए आवश्यक पुस्तकालयों और मॉड्यूल आयात करें।


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## व्यय मॉडल परिभाषित करें

व्यक्तिगत व्ययों के लिए एक Pydantic मॉडल बनाएं और एक ExpenseFormatter क्लास बनाएं जो उपयोगकर्ता प्रश्न को संरचित व्यय डेटा में परिवर्तित करे।

प्रत्येक व्यय इस प्रारूप में प्रस्तुत किया जाएगा:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## टूल्स को परिभाषित करना - ईमेल जनरेट करना

एक टूल फंक्शन बनाएं जो खर्च के दावे को सबमिट करने के लिए ईमेल जनरेट करे।
- यह टूल Microsoft Agent Framework के `@tool` डेकोरेटर का उपयोग करता है।
- यह खर्चों की कुल राशि की गणना करता है और विवरण को ईमेल के बॉडी में फॉर्मेट करता है।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# रसीद छवियों से यात्रा व्यय निकालने का उपकरण

एक उपकरण फ़ंक्शन बनाएं जो रसीद छवियों से यात्रा व्यय निकाले।
- यह उपकरण Microsoft Agent Framework के `@tool` डेकोरेटर का उपयोग करता है।
- यह रसीद छवि को पढ़ता है, इसे बेस64 के रूप में एन्कोड करता है, और एजेंट के विश्लेषण के लिए डेटा URI लौटाता है।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

एजेंट्स को परिभाषित करें और उन्हें `WorkflowBuilder` का उपयोग करके एक अनुक्रमिक कार्यप्रवाह में वायर करें।
- OCR एजेंट रसीद छवि से संरचित खर्च डेटा निकालने के लिए `load_receipt_image` टूल का उपयोग करता है।
- ईमेल एजेंट निकाले गए डेटा को लेता है और `generate_expense_email` टूल का उपयोग करके एक पेशेवर खर्च दावा ईमेल बनाता है।
- `WorkflowBuilder` के साथ `add_edge` एक अनुक्रमिक पाइपलाइन बनाता है: OCR एजेंट → ईमेल एजेंट।


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

क्रमिक वर्कफ़्लो का निर्माण करें और इसे रसीद की छवि को संसाधित करने और खर्च का दावा ईमेल उत्पन्न करने के लिए चलाएं।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
इस दस्तावेज़ का अनुवाद एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़, जो अपनी मूल भाषा में है, उसे अधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
